Keras Seq2Seq 전체 예제 (Toy 데이터 포함)

이 코드는 아주 작은 번역 데이터 + Seq2Seq 학습 전체 흐름을 한 번에 보여주는 예제이다.

Encoder Input:  i am student
        ↓
    LSTM compress
        ↓
   state_h, state_c
        ↓
Decoder Input: <start> 나는 학생이다
        ↓
    LSTM generate
        ↓
Output: 나는 학생이다 <end>


Encoder = 문장 이해
Decoder = 문장 생성
Teacher forcing = 정답을 넣고 학습
Tokenizer = 단어 → 숫자 변환

In [1]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:
# 2. Toy 데이터 (영어 → 한국어)
input_texts = [
    "i am student",
    "i love you"
]

target_texts = [
    "<start> 나는 학생이다 <end>",
    "<start> 나는 너를 사랑한다 <end>"
]

In [ ]:
#3. Tokenizer (단어 → 숫자)

# 입력 tokenizer
input_tokenizer = Tokenizer()
input_tokenizer.fit_on_texts(input_texts)
input_seq = input_tokenizer.texts_to_sequences(input_texts)

# 출력 tokenizer
output_tokenizer = Tokenizer(filters='')
output_tokenizer.fit_on_texts(target_texts)
target_seq = output_tokenizer.texts_to_sequences(target_texts)

In [ ]:
# 4. 패딩
# 👉 뭐 하는 코드냐?
# 각 문장의 길이 중 가장 긴 길이를 찾는 것
# 👉 왜 필요하냐?

# 👉 딥러닝 모델은 모든 입력 길이가 같아야 함
# input_seq = [
#     [3, 4, 5],        # 길이 3
#     [3, 6, 7, 8]      # 길이 4
# ]
max_input_len = max(len(seq) for seq in input_seq)
max_target_len = max(len(seq) for seq in target_seq)

# 👉 의미
# 짧은 문장 뒤에 0을 붙여서 길이를 맞춤
# 📌 결과 예시
# input_seq = [
#     [3, 4, 5],
#     [3, 6, 7, 8]
# ]

# 👉 패딩 결과:

# encoder_input_data = [
#     [3, 4, 5, 0],
#     [3, 6, 7, 8]
# ]
encoder_input_data = pad_sequences(input_seq, maxlen=max_input_len, padding='post')

#3️⃣ decoder_input_data (Teacher Forcing 핵심)
#👉 핵심 개념
#👉 정답 문장을 한 칸 왼쪽으로 밀어서 입력으로 사용
# target_seq = [
#     [1, 8, 9, 2]   # <start> 나는 학생이다 <end>
# ]
# 👉 seq[:-1]
# [1, 8, 9]
# <start> 나는 학생이다
# 👉 왜 이렇게 하냐?

# 👉 Decoder는 이렇게 학습됨:

# 입력: <start> → 나는 → 학생이다
# 출력:       나는 → 학생이다 → <end>
decoder_input_data = pad_sequences([seq[:-1] for seq in target_seq], maxlen=max_target_len-1, padding='post')

# 4️⃣ decoder_target_data (정답)
# 👉 seq[1:]
# [8, 9, 2]

# 👉 의미:
# 나는 학생이다 <end>
decoder_target_data = pad_sequences([seq[1:] for seq in target_seq], maxlen=max_target_len-1, padding='post')
# 🔥 5️⃣ 핵심 관계 (진짜 중요)
# decoder_input_data   → <start> 나는 학생이다
# decoder_target_data  →        나는 학생이다 <end>

# 👉 한 칸씩 밀려 있음

# 🔥 6️⃣ 왜 이렇게 해야 되냐?

# 👉 모델이 이렇게 학습되도록:

# 입력: <start>
# 출력: 나는

# 입력: 나는
# 출력: 학생이다

# 입력: 학생이다
# 출력: <end>
#👉 즉, 다음 단어 예측 학습

# 🚀 핵심 요약 (진짜 중요)
# Encoder 입력 = 원문

# Decoder 입력 = 정답 (마지막 제거)
# Decoder 정답 = 정답 (처음 제거)
print(encoder_input_data)
print(decoder_input_data)
print(decoder_target_data)


[[1 2 3]
 [1 4 5]]
[[1 2 4 0]
 [1 2 5 6]]
[[2 4 3 0]
 [2 5 6 3]]


🔥 1️⃣ 먼저 가정 (단어 사전)

네 결과를 보면 이런 구조로 매핑된 걸로 볼 수 있어:

📌 입력(Encoder) 쪽
1 → i
2 → am
3 → student
4 → love
5 → you
📌 출력(Decoder) 쪽
1 → <start>
2 → 나는
3 → <end>
4 → 학생이다
5 → 너를
6 → 사랑한다
🔥 2️⃣ encoder_input_data 해석
[[1 2 3]
 [1 4 5]]

👉 단어로 바꾸면:

[ i     am     student ]
[ i     love   you     ]
🔥 3️⃣ decoder_input_data 해석
[[1 2 4 0]
 [1 2 5 6]]

👉 단어로 바꾸면:

[ <start>   나는   학생이다   (padding) ]
[ <start>   나는   너를     사랑한다 ]

👉 여기서 0 = padding

🔥 4️⃣ decoder_target_data 해석
[[2 4 3 0]
 [2 5 6 3]]

👉 단어로 바꾸면:

[ 나는   학생이다   <end>   (padding) ]
[ 나는   너를     사랑한다   <end> ]
🔥 5️⃣ 한눈에 비교 (핵심)
decoder_input_data      decoder_target_data

<start>   나는   학생이다   →   나는   학생이다   <end>
<start>   나는   너를     사랑한다 →   나는   너를     사랑한다   <end>

👉 👉 👉 한 칸 shift 구조

🔥 6️⃣ 진짜 중요한 의미

첫 번째 문장 기준:

입력: <start> → 출력: 나는
입력: 나는    → 출력: 학생이다
입력: 학생이다 → 출력: <end>
💡 핵심 요약
encoder_input_data = 영어 문장
decoder_input_data = <start>부터 시작하는 정답
decoder_target_data = 다음 단어 정답
🚀 한 줄 핵심

👉 Seq2Seq는 "이전 단어 → 다음 단어" 예측을 반복하는 구조다

In [6]:
# 5. 하이퍼파라미터
vocab_size_enc = len(input_tokenizer.word_index) + 1
vocab_size_dec = len(output_tokenizer.word_index) + 1
latent_dim = 256

🔥 1️⃣
vocab_size_enc = len(input_tokenizer.word_index) + 1
👉 의미
Encoder 입력 단어 개수 (어휘 수)
👉 왜 +1?
Tokenizer는 보통 1번부터 단어를 인덱싱함
0은 padding(빈칸)으로 따로 쓰기 때문
📌 예시
input_tokenizer.word_index = {
    'i': 1,
    'am': 2,
    'student': 3,
    'love': 4,
    'you': 5
}

👉 길이:

len(...) = 5

👉 실제 vocab size:

vocab_size_enc = 6   # (0 포함)
💡 왜 중요?

👉 Embedding에서 사용됨

Embedding(input_dim=vocab_size_enc, ...)

즉:

0 ~ 5까지 총 6개의 단어를 표현
🔥 2️⃣
vocab_size_dec = len(output_tokenizer.word_index) + 1
👉 의미
Decoder 출력 단어 개수
📌 예시
output_tokenizer.word_index = {
    '<start>': 1,
    '나는': 2,
    '<end>': 3,
    '학생이다': 4,
    '너를': 5,
    '사랑한다': 6
}

👉 결과:

vocab_size_dec = 7
💡 왜 중요?

👉 마지막 Dense layer에서 사용

Dense(vocab_size_dec, activation='softmax')

👉 의미:

"다음 단어 후보를 7개 중에서 선택"
🔥 3️⃣
latent_dim = 256
👉 의미
LSTM의 hidden state 크기 (기억 용량)
💡 직관적으로
문장 → [256차원 벡터] → 의미 압축
📌 예시
"I am student"
→ [0.23, -0.91, 0.77, ..., 256개 값]

👉 이게 바로 context vector (문장 의미)

💡 크기 영향
값	의미
작다 (32, 64)	빠름, 성능 낮음
중간 (128, 256)	보통
크다 (512, 1024)	성능↑, 느림

🔥 전체 연결
Encoder:
단어 개수 (vocab_size_enc) → Embedding → LSTM(latent_dim)

Decoder:
LSTM(latent_dim) → Dense(vocab_size_dec)
🚀 핵심 요약
vocab_size_enc = 입력 단어 종류 수 (+ padding)
vocab_size_dec = 출력 단어 종류 수 (+ padding)
latent_dim = 문장을 압축하는 벡터 크기
💡 한 줄 핵심

👉
vocab_size = "단어 종류 수"
latent_dim = "문장 이해 능력 크기"

In [7]:
# 6. Encoder
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(vocab_size_enc, latent_dim)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

🔥 1️⃣ Encoder 입력
encoder_inputs = Input(shape=(None,))

👉 의미
Encoder에 들어오는 입력 문장
None → 문장 길이는 가변

📌 예시
"I am student"
→ [1, 2, 3]

🔥 2️⃣ Embedding
enc_emb = Embedding(vocab_size_enc, latent_dim)(encoder_inputs)
👉 의미
단어 → 벡터 변환
📌 흐름
[1, 2, 3]
   ↓
Embedding
   ↓
[
 [0.2, 0.1, ...],   # i
 [0.5, 0.3, ...],   # am
 [0.9, 0.7, ...]    # student
]

👉 shape:

(문장길이, 256)
🔥 3️⃣ LSTM 생성
encoder_lstm = LSTM(latent_dim, return_state=True)
👉 의미
시퀀스를 처리하는 RNN
return_state=True → 마지막 상태를 반환
🔥 4️⃣ 문장 압축 (핵심)
_, state_h, state_c = encoder_lstm(enc_emb)
👉 의미
state_h → 현재 상태 (short-term memory)
state_c → 장기 기억 (long-term memory)

👉 둘 다 “문장 의미”를 담고 있음

📌 그림으로 이해
입력 단어:
I → am → student

        ↓
     LSTM 통과
        ↓

state_h = [0.2, 0.8, ..., 256]
state_c = [0.5, 0.1, ..., 256]

👉 이게 바로:

문장 전체 의미 요약 벡터
❗ _ (underscore)의 의미
_, state_h, state_c

👉 첫 번째 값(전체 출력)은 안 쓰겠다는 뜻

🔥 5️⃣ Decoder로 전달
encoder_states = [state_h, state_c]
👉 의미
Encoder가 만든 “문장 의미”를 Decoder로 넘김
🔥 전체 흐름 그림
[입력 문장]
I → am → student
        ↓
    Embedding
        ↓
    LSTM Encoder
        ↓
------------------------
 state_h (현재 기억)
 state_c (장기 기억)
------------------------
        ↓
   Decoder로 전달
💡 핵심 요약
Encoder = 문장을 읽고 → 하나의 벡터로 압축
🚀 진짜 중요한 포인트

👉 이 구조의 문제:

모든 정보를 state 하나에 압축
→ 긴 문장에서 정보 손실 발생

그래서 등장한 게:

Attention (중요🔥)

In [8]:
# Decoder
decoder_inputs = Input(shape=(None,))
dec_emb = Embedding(vocab_size_dec, latent_dim)(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(vocab_size_dec, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

🔥 1️⃣ Decoder 입력
decoder_inputs = Input(shape=(None,))
👉 의미
Decoder에 들어가는 입력 문장
📌 학습 시 입력
<start> 나는 학생이다

👉 숫자로:

[1, 2, 4]
🔥 2️⃣ Embedding
dec_emb = Embedding(vocab_size_dec, latent_dim)(decoder_inputs)
👉 의미
단어 → 벡터 변환
📌 변환 과정
[1, 2, 4]
   ↓
Embedding
   ↓
[
 [0.1, 0.3, ...],   # <start>
 [0.5, 0.2, ...],   # 나는
 [0.9, 0.7, ...]    # 학생이다
]

👉 shape:

(문장 길이, 256)
🔥 3️⃣ Decoder LSTM 생성
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
👉 옵션 설명
return_sequences=True
👉 모든 시점 출력 (문장 생성해야 하니까 필수)
return_state=True
👉 상태도 같이 반환

🔥 4️⃣ 핵심 부분 (Encoder 연결)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
👉 의미 (🔥 핵심)

👉 Decoder 시작할 때 초기 상태를 Encoder에서 가져옴

encoder_states = [state_h, state_c]
📌 흐름
Encoder:
"I am student"
   ↓
(state_h, state_c)

Decoder 시작:
초기 상태 = (state_h, state_c)

👉 즉:

"이 문장의 의미를 기반으로 번역 시작"
🔥 5️⃣ decoder_outputs 의미
각 시간 step마다 나온 결과 벡터

예:

[
 [0.2, 0.8, ...],
 [0.5, 0.1, ...],
 [0.9, 0.3, ...]
]

👉 아직 “단어”가 아니라 중간 벡터

🔥 6️⃣ Dense (출력층)
decoder_dense = Dense(vocab_size_dec, activation='softmax')
👉 의미
단어 확률로 변환
📌 예시
[0.2, 0.8, ...]
   ↓
softmax
   ↓
"나는": 0.7
"너는": 0.1
...
🔥 7️⃣ 최종 출력
decoder_outputs = decoder_dense(decoder_outputs)
👉 결과
각 시점마다
"다음 단어 확률"
🔥 전체 흐름 (진짜 중요)
Decoder 입력:
<start> → 나는 → 학생이다

        ↓
Embedding
        ↓
LSTM (Encoder 상태 기반)
        ↓
Dense (softmax)
        ↓
출력:
나는 → 학생이다 → <end>
🔥 시간 흐름으로 보면
Step 1
입력: <start>
출력: 나는

Step 2
입력: 나는
출력: 학생이다

Step 3
입력: 학생이다
출력: <end>
💡 핵심 요약
Decoder = 이전 단어 + Encoder 의미 → 다음 단어 생성
🚀 Encoder vs Decoder 비교
역할	Encoder	Decoder
입력	원문	정답 문장
역할	압축	생성
출력	state	단어 확률
💥 한 줄 핵심

👉
Decoder는 "이전 단어를 보고 다음 단어를 맞추는 기계"

In [ ]:
#8. 모델 생성

# 👉 의미
# Keras Functional API로 모델을 정의
# 입력이 2개, 출력이 1개인 구조

# 입력 1: encoder_inputs  → 원문 (영어)
# 입력 2: decoder_inputs  → 정답 문장 (shift된 형태)

# 출력: decoder_outputs → 다음 단어 확률

# encoder_inputs ─────┐
#                     ├──→ Seq2Seq 모델 → decoder_outputs
# decoder_inputs ─────┘

# model = 구조 정의
# compile = 학습 방식 정의
# summary = 구조 확인

# 💥 진짜 핵심 이해

# 👉 이 모델은 이렇게 학습됨:

# 입력:
# (I am student, <start> 나는 학생이다)

# 출력:
# (나는 학생이다 <end>)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.summary()

In [15]:
# 🔥 1️⃣ 지금 문제 상황

# 우리가 만든 데이터 형태를 보자.

# 👉 현재 decoder_target_data
# [[8, 9, 2],
#  [8, 10, 11]]

# 👉 shape:

# (batch_size, sequence_length)
# 예: (2, 3)
# ❗ 그런데 모델 출력은?

# Decoder 출력은 이렇게 생김:

# (batch_size, sequence_length, vocab_size)
# 예: (2, 3, 10000)

# 👉 즉:

# 예측값 = 3차원
# 정답값 = 2차원

# 👉 ❌ shape 안 맞음

# 🔥 2️⃣ expand_dims 역할
# np.expand_dims(decoder_target_data, -1)

# 👉 마지막에 차원을 하나 추가

# 📌 결과
# [
#  [[8], [9], [2]],
#  [[8], [10], [11]]
# ]

# 👉 shape:

# (2, 3, 1)
# 🔥 3️⃣ 왜 필요한가?

# 👉 loss 함수 때문

# loss='sparse_categorical_crossentropy'

# 이 loss는 이렇게 비교함:

# 예측: (batch, seq_len, vocab_size)
# 정답: (batch, seq_len, 1)

# 👉 즉:

# 각 위치마다 "정답 인덱스 하나" 필요
# 🔥 4️⃣ 직관적으로 이해
# 예측:
# [나는, 학생이다, <end> 확률]

# 정답:
# [나는, 학생이다, <end> 인덱스]

# 👉 그래서 이렇게 맞춰줌:

# 정답 → [[2], [4], [3]]
# 🔥 5️⃣ 그림으로 이해
# Before:
# decoder_target_data
# (2, 3)
# [8  9  2]

# After:
# (2, 3, 1)
# [[8]
#  [9]
#  [2]]
# 🔥 6️⃣ 한 줄 핵심
# 정답을 "단어 하나짜리 벡터" 형태로 바꿔서
# 모델 출력과 shape 맞추는 작업
# 💡 추가 팁

# 👉 만약 expand_dims 안 하면?

# ValueError: shape mismatch
# 🚀 최종 요약
# expand_dims = 차원 하나 추가해서
# loss 계산 가능하게 만드는 작업

#9. reshape (loss용)
decoder_target_data = np.expand_dims(decoder_target_data, -1)

In [ ]:
# 🔥 1️⃣ 전체 의미

# 👉 한 줄 요약:

# (입력 데이터) → 모델 → (정답과 비교) → 반복 학습
# 🔥 2️⃣ 입력 데이터 부분
# [encoder_input_data, decoder_input_data]
# 👉 의미

# 입력이 2개라서 리스트로 묶음

# 📌 각각 역할
# ✔ encoder_input_data
# 영어 문장
# 예: [i, am, student]

# 👉 역할:

# 문장 의미 이해
# ✔ decoder_input_data
# <start> 나는 학생이다

# 👉 역할:

# 문장 생성 시작 + teacher forcing
# 🔥 3️⃣ 정답 데이터
# decoder_target_data
# 👉 의미
# 나는 학생이다 <end>

# 👉 모델이 맞춰야 할 정답

# 🔥 4️⃣ 내부에서 실제로 일어나는 일
# 👉 Step 단위로 보면
# 입력:
# encoder_input = I am student
# decoder_input = <start> 나는 학생이다

# 정답:
# 나는 학생이다 <end>
# 👉 모델 동작
# Step 1
# 입력: <start>
# 예측: 나는
# 정답: 나는

# Step 2
# 입력: 나는
# 예측: 학생이다
# 정답: 학생이다

# Step 3
# 입력: 학생이다
# 예측: <end>
# 정답: <end>
# 👉 핵심

# 이전 단어 → 다음 단어 맞추기
# 🔥 5️⃣ loss 계산
# 예측 vs 정답 비교

# 예:

# 예측: 나는 (0.7 확률)
# 정답: 나는
# → loss 작음

# 예측: 너는 (0.8 확률)
# 정답: 나는
# → loss 큼
# 🔥 6️⃣ 가중치 업데이트
# 틀리면 → 수정
# 맞으면 → 유지

# 👉 이걸 optimizer='adam'이 수행

# 🔥 7️⃣ epochs=50
# epochs=50
# 👉 의미
# 전체 데이터를 50번 반복 학습

# 📌 왜 여러 번?
# 1번 → 대충 배움
# 50번 → 점점 정확해짐

# 🔥 8️⃣ 전체 흐름
# 1. 입력 넣기
# 2. 예측하기
# 3. 정답과 비교
# 4. loss 계산
# 5. 가중치 수정
# 6. 반복 (50번)
# 💡 핵심 요약
# model.fit = 모델을 "문제 풀면서 공부시키는 과정"
# 💥 한 줄 핵심

# 👉
# Seq2Seq 학습 = “이전 단어 → 다음 단어” 맞추기 반복
#학습
model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    epochs=50
)

In [ ]:
# Keras Seq2Seq 전체 예제 (Toy 데이터 포함)

# 이 코드는 **아주 작은 번역 데이터 + Seq2Seq 학습 전체 흐름**을 한 번에 보여주는 예제이다.
# ---
# # 1. Import

import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


# 2. Toy 데이터 (영어 → 한국어)
input_texts = [
    "i am student",
    "i love you"
]

target_texts = [
    "<start> 나는 학생이다 <end>",
    "<start> 나는 너를 사랑한다 <end>"
]

# 3. Tokenizer (단어 → 숫자)


# 입력 tokenizer
input_tokenizer = Tokenizer()
input_tokenizer.fit_on_texts(input_texts)
input_seq = input_tokenizer.texts_to_sequences(input_texts)

# 출력 tokenizer
output_tokenizer = Tokenizer(filters='')
output_tokenizer.fit_on_texts(target_texts)
target_seq = output_tokenizer.texts_to_sequences(target_texts)


# 4. 패딩


max_input_len = max(len(seq) for seq in input_seq)
max_target_len = max(len(seq) for seq in target_seq)

encoder_input_data = pad_sequences(input_seq, maxlen=max_input_len, padding='post')
decoder_input_data = pad_sequences([seq[:-1] for seq in target_seq], maxlen=max_target_len-1, padding='post')
decoder_target_data = pad_sequences([seq[1:] for seq in target_seq], maxlen=max_target_len-1, padding='post')

# 5. 하이퍼파라미터
vocab_size_enc = len(input_tokenizer.word_index) + 1
vocab_size_dec = len(output_tokenizer.word_index) + 1
latent_dim = 256


# 6. Encoder
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(vocab_size_enc, latent_dim)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# 7. Decoder


decoder_inputs = Input(shape=(None,))
dec_emb = Embedding(vocab_size_dec, latent_dim)(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(vocab_size_dec, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)


# 8. 모델 생성


model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.summary()


# 9. reshape (loss용)

decoder_target_data = np.expand_dims(decoder_target_data, -1)

# 10. 학습


model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    epochs=50
)


# 11. 전체 흐름

# ```text
# Encoder Input:  i am student
#         ↓
#     LSTM compress
#         ↓
#    state_h, state_c
#         ↓
# Decoder Input: <start> 나는 학생이다
#         ↓
#     LSTM generate
#         ↓
# Output: 나는 학생이다 <end>
# ```

# 12. 핵심

# * Encoder = 문장 이해
# * Decoder = 문장 생성
# * Teacher forcing = 정답을 넣고 학습
# * Tokenizer = 단어 → 숫자 변환

# # 🚀 확장

# 이 코드는 기본 구조이며 실제 성능 개선은:

# * Attention 추가
# * Transformer 사용


# 13. Inference (번역 테스트 코드)

#학습이 끝난 후에는 **Encoder/Decoder를 분리해서** 한 단어씩 생성해야 한다.

# --- Encoder 모델 (inference용)
encoder_model = Model(encoder_inputs, encoder_states)

# --- Decoder 모델 (inference용)
# 상태 입력
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# Decoder embedding (학습 때 사용한 레이어 재사용)
dec_emb2 = Embedding(vocab_size_dec, latent_dim)(decoder_inputs)

# LSTM (이전 상태를 입력으로 받음)
decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    dec_emb2, initial_state=decoder_states_inputs
)
decoder_states2 = [state_h2, state_c2]

# Dense
decoder_outputs2 = decoder_dense(decoder_outputs2)

# 최종 decoder 모델
decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2] + decoder_states2
)


# 14. 디코딩 함수 (문장 생성)

# 인덱스 → 단어
reverse_target_word_index = {i: w for w, i in output_tokenizer.word_index.items()}

start_token = output_tokenizer.word_index['<start>']
end_token = output_tokenizer.word_index['<end>']


def decode_sequence(input_seq):
    # 1) Encoder로 상태 추출
    states_value = encoder_model.predict(input_seq)

    # 2) 시작 토큰으로 초기화
    target_seq = np.array([[start_token]])

    decoded_sentence = ''

    # 3) 한 단어씩 생성
    for _ in range(20):  # 최대 길이 제한
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

        # 가장 확률 높은 단어 선택
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_target_word_index.get(sampled_token_index, '')

        if sampled_word == '<end>' or sampled_word == '':
            break

        decoded_sentence += ' ' + sampled_word

        # 다음 입력으로 사용
        target_seq = np.array([[sampled_token_index]])

        # 상태 업데이트
        states_value = [h, c]

    return decoded_sentence.strip()

# 15. 테스트


# 테스트 문장
input_text = "i am student"

# 토큰화
seq = input_tokenizer.texts_to_sequences([input_text])
seq = pad_sequences(seq, maxlen=max_input_len, padding='post')

# 번역 실행
result = decode_sequence(seq)
print("입력:", input_text)
print("출력:", result)


# 16. 동작 흐름

# ```text
# 입력 문장 → Encoder → 상태 벡터
#         ↓
# <start> 입력
#         ↓
# Decoder → 단어 생성
#         ↓
# 다음 단어를 다시 입력
#         ↓
# <end> 나오면 종료
# ```


# ⚠️ 주의

# * Toy 데이터라 결과 품질은 낮음 (데이터가 너무 적음)
# * 실제 성능을 위해서는 데이터 + Attention 필수


# Keras Seq2Seq 전체 예제 (Toy 데이터 포함)

이 코드는 **아주 작은 번역 데이터 + Seq2Seq 학습 전체 흐름**을 한 번에 보여주는 예제이다.

---

# 1. Import

```python
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
```

---

# 2. Toy 데이터 (영어 → 한국어)

```python
input_texts = [
    "i am student",
    "i love you"
]

target_texts = [
    "<start> 나는 학생이다 <end>",
    "<start> 나는 너를 사랑한다 <end>"
]
```

---

# 3. Tokenizer (단어 → 숫자)

```python
# 입력 tokenizer
input_tokenizer = Tokenizer()
input_tokenizer.fit_on_texts(input_texts)
input_seq = input_tokenizer.texts_to_sequences(input_texts)

# 출력 tokenizer
output_tokenizer = Tokenizer(filters='')
output_tokenizer.fit_on_texts(target_texts)
target_seq = output_tokenizer.texts_to_sequences(target_texts)
```

---

# 4. 패딩

```python
max_input_len = max(len(seq) for seq in input_seq)
max_target_len = max(len(seq) for seq in target_seq)

encoder_input_data = pad_sequences(input_seq, maxlen=max_input_len, padding='post')
decoder_input_data = pad_sequences([seq[:-1] for seq in target_seq], maxlen=max_target_len-1, padding='post')
decoder_target_data = pad_sequences([seq[1:] for seq in target_seq], maxlen=max_target_len-1, padding='post')
```

---

# 5. 하이퍼파라미터

```python
vocab_size_enc = len(input_tokenizer.word_index) + 1
vocab_size_dec = len(output_tokenizer.word_index) + 1
latent_dim = 256
```

---

# 6. Encoder

```python
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(vocab_size_enc, latent_dim)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]
```

---

# 7. Decoder

```python
decoder_inputs = Input(shape=(None,))
dec_emb = Embedding(vocab_size_dec, latent_dim)(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(vocab_size_dec, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)
```

---

# 8. 모델 생성

```python
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.summary()
```

---

# 9. reshape (loss용)

```python
decoder_target_data = np.expand_dims(decoder_target_data, -1)
```

---

# 10. 학습

```python
model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    epochs=50
)
```

---

# 11. 전체 흐름

```text
Encoder Input:  i am student
        ↓
    LSTM compress
        ↓
   state_h, state_c
        ↓
Decoder Input: <start> 나는 학생이다
        ↓
    LSTM generate
        ↓
Output: 나는 학생이다 <end>
```

---

# 12. 핵심

* Encoder = 문장 이해
* Decoder = 문장 생성
* Teacher forcing = 정답을 넣고 학습
* Tokenizer = 단어 → 숫자 변환

---

# 🚀 확장

이 코드는 기본 구조이며 실제 성능 개선은:

* Attention 추가
* Transformer 사용

---

# 13. Inference (번역 테스트 코드)

학습이 끝난 후에는 **Encoder/Decoder를 분리해서** 한 단어씩 생성해야 한다.

```python
# --- Encoder 모델 (inference용)
encoder_model = Model(encoder_inputs, encoder_states)

# --- Decoder 모델 (inference용)
# 상태 입력
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# Decoder embedding (학습 때 사용한 레이어 재사용)
dec_emb2 = Embedding(vocab_size_dec, latent_dim)(decoder_inputs)

# LSTM (이전 상태를 입력으로 받음)
decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    dec_emb2, initial_state=decoder_states_inputs
)
decoder_states2 = [state_h2, state_c2]

# Dense
decoder_outputs2 = decoder_dense(decoder_outputs2)

# 최종 decoder 모델
decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2] + decoder_states2
)
```

---

# 14. 디코딩 함수 (문장 생성)

```python
# 인덱스 → 단어
reverse_target_word_index = {i: w for w, i in output_tokenizer.word_index.items()}

start_token = output_tokenizer.word_index['<start>']
end_token = output_tokenizer.word_index['<end>']


def decode_sequence(input_seq):
    # 1) Encoder로 상태 추출
    states_value = encoder_model.predict(input_seq)

    # 2) 시작 토큰으로 초기화
    target_seq = np.array([[start_token]])

    decoded_sentence = ''

    # 3) 한 단어씩 생성
    for _ in range(20):  # 최대 길이 제한
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

        # 가장 확률 높은 단어 선택
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_target_word_index.get(sampled_token_index, '')

        if sampled_word == '<end>' or sampled_word == '':
            break

        decoded_sentence += ' ' + sampled_word

        # 다음 입력으로 사용
        target_seq = np.array([[sampled_token_index]])

        # 상태 업데이트
        states_value = [h, c]

    return decoded_sentence.strip()
```

---

# 15. 테스트

```python
# 테스트 문장
input_text = "i am student"

# 토큰화
seq = input_tokenizer.texts_to_sequences([input_text])
seq = pad_sequences(seq, maxlen=max_input_len, padding='post')

# 번역 실행
result = decode_sequence(seq)
print("입력:", input_text)
print("출력:", result)
```

---

# 16. 동작 흐름

```text
입력 문장 → Encoder → 상태 벡터
        ↓
<start> 입력
        ↓
Decoder → 단어 생성
        ↓
다음 단어를 다시 입력
        ↓
<end> 나오면 종료
```

---

# ⚠️ 주의

* Toy 데이터라 결과 품질은 낮음 (데이터가 너무 적음)
* 실제 성능을 위해서는 데이터 + Attention 필수
